In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

In [2]:
import shutil
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

/home/usuario-utp/anaconda3/envs/yessica/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class InspectableTransformerEncoder(nn.Module):
    """
    Réplica en idea del InspectableTransformerEncoder de TensorFlow/Keras.

    Flujo:
      input [B, T, C]
        -> proyección opcional a d_model
        -> MultiHeadAttention
        -> residual + LayerNorm
        -> FeedForward
        -> residual + LayerNorm

    Guarda los últimos attention scores para inspección.
    """
    def __init__(
        self,
        in_channels,
        d_model=None,
        num_heads=2,
        intermediate_dim=128,
        dropout=0.1,
        use_input_projection=True,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.d_model = d_model if d_model is not None else in_channels
        self.num_heads = num_heads
        self.intermediate_dim = intermediate_dim
        self.dropout_rate = dropout
        self.use_input_projection = use_input_projection

        if self.d_model % self.num_heads != 0:
            raise ValueError(
                f"d_model={self.d_model} debe ser divisible entre num_heads={self.num_heads}"
            )

        # Proyección opcional para imitar tu adaptación en PyTorch
        if use_input_projection or self.d_model != in_channels:
            self.input_proj = nn.Linear(in_channels, self.d_model)
        else:
            self.input_proj = nn.Identity()

        # Self-attention inspeccionable
        self.self_attention = nn.MultiheadAttention(
            embed_dim=self.d_model,
            num_heads=self.num_heads,
            dropout=self.dropout_rate,
            batch_first=True,
        )

        # Normalización y dropout, igual idea que Keras
        self.self_attention_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)
        self.self_attention_dropout = nn.Dropout(self.dropout_rate)

        # Feedforward
        self.feedforward_intermediate_dense = nn.Linear(self.d_model, self.intermediate_dim)
        self.feedforward_output_dense = nn.Linear(self.intermediate_dim, self.d_model)
        self.feedforward_dropout = nn.Dropout(self.dropout_rate)
        self.feedforward_layer_norm = nn.LayerNorm(self.d_model, eps=1e-6)

        # Para guardar los scores
        self._last_attention_scores = None

    def forward(self, inputs, training=False, need_weights=True, attn_mask=None):
        """
        inputs: [B, T, C]
        returns: [B, T, d_model]
        """
        x = self.input_proj(inputs)  # [B, T, d_model]

        attention_output, attention_scores = self.self_attention(
            query=x,
            key=x,
            value=x,
            attn_mask=attn_mask,
            need_weights=need_weights,
            average_attn_weights=False,  # devuelve [B, H, T, T]
        )

        if need_weights:
            self._last_attention_scores = attention_scores
        else:
            self._last_attention_scores = None

        attention_output = self.self_attention_dropout(attention_output)
        attention_output = self.self_attention_layer_norm(x + attention_output)

        ff_output = self.feedforward_intermediate_dense(attention_output)
        ff_output = F.gelu(ff_output)
        ff_output = self.feedforward_output_dense(ff_output)
        ff_output = self.feedforward_dropout(ff_output)

        output = self.feedforward_layer_norm(attention_output + ff_output)
        return output

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado attention scores todavía. "
                "Haz un forward pass con need_weights=True."
            )
        return self._last_attention_scores

    def get_attention_weights(self):
        """
        Intenta imitar la idea de inspeccionar pesos Q, K, V y O.
        En PyTorch MultiheadAttention, Q/K/V suelen venir concatenados en in_proj_weight.
        """
        attn = self.self_attention

        weights_dict = {
            "in_proj_weight": attn.in_proj_weight.detach().cpu(),
            "in_proj_bias": attn.in_proj_bias.detach().cpu() if attn.in_proj_bias is not None else None,
            "out_proj_weight": attn.out_proj.weight.detach().cpu(),
            "out_proj_bias": attn.out_proj.bias.detach().cpu() if attn.out_proj.bias is not None else None,
        }
        return weights_dict

In [5]:
# =========================================================
# 3) FILTRO TEMPORAL POR CANAL  φ(.)
#    Réplica del bloque TF:
#    DepthwiseConv1D -> ReLU -> AvgPool1D -> BatchNorm1d
# =========================================================
class ChannelwiseTemporalFilter(nn.Module):
    def __init__(self, channels, kernel_size=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=kernel_size,
                stride=1,
                padding=0,          # TF: padding='valid'
                groups=channels,    # depthwise
                bias=True           # TF DepthwiseConv1D por defecto usa bias=True
            ),
            nn.ReLU(),
            nn.AvgPool1d(
                kernel_size=4,
                stride=4
            ),
            nn.BatchNorm1d(channels)
        )

    def forward(self, x):
        # x: [B, C, T]
        return self.net(x)

# =========================================================
# 4) TAKENS CONV1D
# =========================================================
class TakensConv1D(nn.Module):
    def __init__(self, dx=5, dy=5, tau=1, mu=4):
        super().__init__()
        self.dx = int(dx)
        self.dy = int(dy)
        self.tau = int(tau)
        self.mu = int(mu)
        self.num_filters = self.dx + self.dy + 1

        # Implementación original
        kernel_size = self.mu + (self.dx - 1) * self.tau + 1
        self.kernel_size = kernel_size

        kernel = torch.zeros(self.num_filters, 1, kernel_size)

        # x_sub_t_minus_mu
        for i in range(self.dx):
            kernel[i, 0, self.mu + i * self.tau] = 1.0

        # y_sub_t_minus_1
        for i in range(self.dy):
            kernel[self.dx + i, 0, (i + 1) * self.tau] = 1.0

        # y_sub_t
        kernel[self.dx + self.dy, 0, 0] = 1.0

        kernel = torch.flip(kernel, dims=[-1])
        self.register_buffer("kernel", kernel)

    def forward(self, inputs):
        """
        inputs: [B, C, T]
        return:
            x_sub_t_minus_mu: [B, C, L, dx]
            y_sub_t_minus_1:  [B, C, L, dy]
            y_sub_t:          [B, C, L, 1]
        """
        B, C, T = inputs.shape

        reshaped = inputs.reshape(B * C, 1, T)

        conv_output = F.conv1d(
            reshaped,
            self.kernel,
            bias=None,
            stride=self.tau,
            padding=0
        )  # [B*C, num_filters, L]

        L = conv_output.shape[-1]
        output = conv_output.reshape(B, C, self.num_filters, L).permute(0, 1, 3, 2)

        x_sub_t_minus_mu = output[..., :self.dx]
        y_sub_t_minus_1 = output[..., self.dx:self.dx + self.dy]
        y_sub_t = output[..., -1:]

        return x_sub_t_minus_mu, y_sub_t_minus_1, y_sub_t


# =========================================================
# 5) KERNEL LAYER
#    Replica tu KernelLayer de TF
# =========================================================
class KernelLayer(nn.Module):
    def __init__(
        self,
        amplitude=1.0,
        trainable_amplitude=False,
        length_scale=1.0,
        trainable_length_scale=False,
        alpha=1.0,
        trainable_alpha=False,
        kernel_type="gaussian",
    ):
        super().__init__()
        self.kernel_type = kernel_type.lower()

        amp = torch.tensor(float(amplitude), dtype=torch.float32)
        ls = torch.tensor(float(length_scale), dtype=torch.float32)
        a = torch.tensor(float(alpha), dtype=torch.float32)

        if trainable_amplitude:
            self.amplitude = nn.Parameter(amp)
        else:
            self.register_buffer("amplitude", amp)

        if trainable_length_scale:
            self.length_scale = nn.Parameter(ls)
        else:
            self.register_buffer("length_scale", ls)

        if self.kernel_type == "rational_quadratic":
            if trainable_alpha:
                self.alpha = nn.Parameter(a)
            else:
                self.register_buffer("alpha", a)

    def forward(self, X):
        """
        X: [B, C, L, D]
        return: kernel.matrix(X, X) ~ [B, C, L, L]
        """
        diff = X.unsqueeze(-2) - X.unsqueeze(-3)   # [B, C, L, L, D]
        dist2 = (diff ** 2).sum(dim=-1)            # [B, C, L, L]

        amp = self.amplitude
        ls = torch.clamp(self.length_scale, min=1e-8)

        if self.kernel_type == "gaussian":
            K = (amp ** 2) * torch.exp(-dist2 / (2.0 * (ls ** 2)))
        elif self.kernel_type == "rational_quadratic":
            alpha = torch.clamp(self.alpha, min=1e-8)
            K = (amp ** 2) * (1.0 + dist2 / (2.0 * alpha * (ls ** 2))) ** (-alpha)
        else:
            raise ValueError(f"Unsupported kernel_type: {self.kernel_type}")

        return K


# =========================================================
# 6) TRANSFER ENTROPY LAYER
# =========================================================
class TransferEntropyLayer(nn.Module):
    def __init__(self, alpha=2):
        super().__init__()
        self.alpha = int(alpha)

    def compute_entropy(self, K_hadamard):
        """
        K_hadamard: [..., N, N]
        """
        trace_hadamard = torch.diagonal(K_hadamard, dim1=-2, dim2=-1).sum(dim=-1)
        trace_hadamard = trace_hadamard.unsqueeze(-1).unsqueeze(-1) + 1e-8

        K_normalized = K_hadamard / trace_hadamard

        K_power = K_normalized @ K_normalized
        trace_power = torch.diagonal(K_power, dim1=-2, dim2=-1).sum(dim=-1)

        if self.alpha == 2:
            H_alpha = -torch.log(trace_power + 1e-8)
        else:
            eigvals = torch.linalg.eigvalsh(K_normalized)
            eigvals = torch.clamp(eigvals.real, min=1e-8)
            H_alpha = torch.log((eigvals ** self.alpha).sum(dim=-1) + 1e-8) / (1 - self.alpha)

        return H_alpha

    def forward(self, K_x, K_y_minus_1, K_y):
        """
        K_x:         [B, C, L, L]
        K_y_minus_1: [B, C, L, L]
        K_y:         [B, C, L, L]

        return TE:   [B, C, C]
        """
        K_x_exp = K_x.unsqueeze(2)               # [B, C, 1, L, L]
        K_y_minus_1_exp = K_y_minus_1.unsqueeze(1)  # [B, 1, C, L, L]
        K_y_exp = K_y.unsqueeze(1)              # [B, 1, C, L, L]

        H_1 = self.compute_entropy(K_y_minus_1_exp * K_x_exp)
        H_2 = self.compute_entropy(K_y_exp * K_y_minus_1_exp * K_x_exp)
        H_3 = self.compute_entropy(K_y_exp * K_y_minus_1_exp)
        H_4 = self.compute_entropy(K_y_minus_1_exp)

        TE = H_1 - H_2 + H_3 - H_4
        return TE


# =========================================================
# 7) REMOVE DIAGONAL FLATTEN
# =========================================================
class RemoveDiagonalFlatten(nn.Module):
    def forward(self, inputs):
        """
        inputs: [B, C, C]
        return: [B, C*(C-1)]
        """
        B, C, C2 = inputs.shape
        if C != C2:
            raise ValueError("RemoveDiagonalFlatten: la matriz de entrada no es cuadrada.")

        mask = ~torch.eye(C, dtype=torch.bool, device=inputs.device)
        result = inputs[:, mask].reshape(B, C * (C - 1))
        return result

In [6]:
# =========================================================
# 8) MODELO HÍBRIDO Transformer + TEKTE
#    X^T -> Transformer -> HWp+bp -> φ -> Takens -> TE -> vec(T-diag(T)) -> clasificador
# =========================================================
class HybridTransformerTEKTE(nn.Module):
    def __init__(
        self,
        chans=19,
        samples=512,
        d_model=64,
        nhead=2,
        num_transformer_layers=1,
        dim_feedforward=128,
        transformer_dropout=0.1,
        phi_kernel_size=9,
        dx=5,
        dy=5,
        tau=1,
        mu=4,
        kernel_type="rational_quadratic",   # o "gaussian"
        kernel_amplitude=1.0,
        kernel_length_scale=1.0,
        kernel_alpha=1.0,
        trainable_kernel_amplitude=False,
        trainable_kernel_length_scale=False,
        trainable_kernel_alpha=False,
        clf_hidden=64,
        clf_dropout=0.3,
        use_pre_transformer_norm=True,
        use_post_transformer_norm=True,
    ):
        super().__init__()
        self.chans = chans
        self.samples = samples
        self.dx = dx
        self.dy = dy
        self.tau = tau
        self.mu = mu
        self.num_transformer_layers = num_transformer_layers
        self.d_model = d_model

        # 1) X: [B, C, T] -> [B, T, C]
        self.pre_transformer_norm = (
            nn.LayerNorm(chans) if use_pre_transformer_norm else nn.Identity()
        )

        # 2) Transformer stack
        self.transformer_layers = nn.ModuleList([
            InspectableTransformerEncoder(
                in_channels=chans if i == 0 else d_model,
                d_model=d_model,
                num_heads=nhead,
                intermediate_dim=dim_feedforward,
                dropout=transformer_dropout,
                use_input_projection=True,
            )
            for i in range(num_transformer_layers)
        ])

        self.post_transformer_norm = (
            nn.LayerNorm(d_model) if use_post_transformer_norm else nn.Identity()
        )

        # 3) projection back to channel space: Xf = H Wp + bp
        self.proj_back = nn.Linear(d_model, chans)

        # 4) φ(.)
        self.phi = ChannelwiseTemporalFilter(
            channels=chans,
            kernel_size=phi_kernel_size
        )

        # 5) Takens
        self.takens = TakensConv1D(dx=dx, dy=dy, tau=tau, mu=mu)

        # 6) Dense projections like your TF model
        self.dense_proj_x = nn.Linear(dx, dx, bias=False)
        self.dense_proj_y1 = nn.Linear(dy, dy, bias=False)
        self.dense_proj_y = nn.Linear(1, 1, bias=False)

        # 7) Kernel layers
        self.kernel_x = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        self.kernel_y_minus_1 = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        self.kernel_y = KernelLayer(
            amplitude=kernel_amplitude,
            trainable_amplitude=trainable_kernel_amplitude,
            length_scale=kernel_length_scale,
            trainable_length_scale=trainable_kernel_length_scale,
            alpha=kernel_alpha,
            trainable_alpha=trainable_kernel_alpha,
            kernel_type=kernel_type,
        )

        # 8) Transfer Entropy
        self.transfer_entropy = TransferEntropyLayer(alpha=2)

        # 9) remove diagonal
        self.remove_diag_flatten = RemoveDiagonalFlatten()

        # 10) classifier
        feat_dim = chans * (chans - 1)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, clf_hidden),
            nn.ReLU(),
            nn.Dropout(clf_dropout),
            nn.Linear(clf_hidden, 1)
        )

    def forward(self, x, return_dict=True, need_attention=False):
        """
        x: [B, C, T]
        """
        B, C, T = x.shape

        if C != self.chans:
            raise ValueError(f"Se esperaban {self.chans} canales y llegaron {C}.")
        if T != self.samples:
            raise ValueError(f"Se esperaban {self.samples} muestras y llegaron {T}.")

        # -------------------------------------------------
        # Hybrid Transformer–TEKTE math:
        # X^T -> Transformer stack -> H -> Xf = HWp + bp -> φ -> Takens -> TE -> classifier
        # -------------------------------------------------

        # (1) transpose: [B, C, T] -> [B, T, C]
        x_t = x.transpose(1, 2)

        # optional norm before transformer
        x_t = self.pre_transformer_norm(x_t)

        # (2) Transformer stack
        H = x_t
        attention_scores_all = [] if need_attention else None

        for layer in self.transformer_layers:
            H = layer(H, need_weights=need_attention)
            if need_attention:
                attention_scores_all.append(layer.get_attention_scores())

        # optional norm after transformer stack
        H = self.post_transformer_norm(H)

        # (3) projection back to channel space
        Xf = self.proj_back(H)                                   # [B, T, C]

        # back to [B, C, T]
        Xf = Xf.transpose(1, 2)

        # (4) nonlinear temporal filter φ
        phi = self.phi(Xf)

        # (5)-(7) Takens states
        x_sub, y_minus_1, y_t = self.takens(phi)

        # dense projections
        x_sub = self.dense_proj_x(x_sub)
        y_minus_1 = self.dense_proj_y1(y_minus_1)
        y_t = self.dense_proj_y(y_t)

        # kernels
        Kx = self.kernel_x(x_sub)
        Ky1 = self.kernel_y_minus_1(y_minus_1)
        Ky = self.kernel_y(y_t)

        # (8)-(10) Transfer Entropy matrix T
        Tmat = self.transfer_entropy(Kx, Ky1, Ky)                # [B, C, C]

        # (11) remove diagonal + flatten
        h = self.remove_diag_flatten(Tmat)

        # (12) classifier logits
        logits = self.classifier(h).squeeze(-1)
        probs = torch.sigmoid(logits)

        if return_dict:
            return {
                "logits": logits,
                "probs": probs,
                "T": Tmat,
                "features": h,
                "Xf": Xf,
                "phi": phi,
                "attention_scores": attention_scores_all if need_attention else None,
            }

        return logits


# =========================================================
# 9) LOSS
# =========================================================
def classification_loss(logits, y_true):
    y_true = y_true.float().view(-1)
    return F.binary_cross_entropy_with_logits(logits, y_true)

In [7]:
model = HybridTransformerTEKTE(
    chans=19,
    samples=512,
    d_model=64,
    nhead=2,
    num_transformer_layers=1,
    dim_feedforward=128,
    transformer_dropout=0.1,
    phi_kernel_size=64,
    dx=5,
    dy=5,
    tau=1,
    mu=4,
    kernel_type="rational_quadratic",   # o "gaussian"
    kernel_amplitude=1.0,
    kernel_length_scale=1.0,
    kernel_alpha=1.0,
    trainable_kernel_amplitude=False,
    trainable_kernel_length_scale=False,
    trainable_kernel_alpha=False,
    clf_hidden=64,
    clf_dropout=0.3,
).to(device)


class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x, return_dict=True, need_attention=False)
        return out["logits"]


wrapped_model = SummaryWrapper(model).to(device)

summary(
    wrapped_model,
    input_size=(1, 19, 512),   # batch, chans, samples
    device=device,
    depth=4,
    col_names=("input_size", "output_size", "num_params", "trainable"),
)

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                                     [1, 19, 512]              [1]                       --                        True
├─HybridTransformerTEKTE: 1-1                      [1, 19, 512]              --                        --                        True
│    └─LayerNorm: 2-1                              [1, 512, 19]              [1, 512, 19]              38                        True
│    └─ModuleList: 2-2                             --                        --                        --                        True
│    │    └─InspectableTransformerEncoder: 3-1     [1, 512, 19]              [1, 512, 64]              --                        True
│    │    │    └─Linear: 4-1                       [1, 512, 19]              [1, 512, 64]              1,280                     True
│    │    │    └─MultiheadAttention: 4-2           --    

In [8]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos), np.array(y, dtype=np.float32), sbjs, window_ids


def get_segmented_data():
    """
    Se tiene que agregar en kaggle la base de datos
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = [archivo[:-4] for archivo in os.listdir(ruta_carpeta_TDAH) if archivo.endswith('.mat')]
    sujetos_TDAH.pop()
    sujetos_control = [archivo[:-4] for archivo in os.listdir(ruta_carpeta_control) if archivo.endswith('.mat')]

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah
    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [9]:
def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


def suggest_model_args(trial, model_name, base_model_args):
    model_name = model_name.lower()

    if model_name != "hybridtransformertekte":
        raise ValueError("Este bloque limpio solo soporta model_name='HybridTransformerTEKTE'.")

    samples = int(base_model_args.get("Samples", 512))

    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    nhead = trial.suggest_categorical("nhead", [1, 2])
    num_transformer_layers = trial.suggest_categorical("num_transformer_layers", [1, 2])
    dim_feedforward = trial.suggest_categorical("dim_feedforward", [64, 128, 256])
    transformer_dropout = trial.suggest_float("transformer_dropout", 0.1, 0.5, step=0.1)

    phi_kernel_size = trial.suggest_int("phi_kernel_size", 3, 125, step=2)

    dx = trial.suggest_int("dx", 1, 10, step=1)
    dy = trial.suggest_int("dy", 1, 10, step=1)
    tau = trial.suggest_int("tau", 1, 5, step=1)
    mu = trial.suggest_int("mu", 0, 10, step=1)

    clf_hidden = trial.suggest_categorical("clf_hidden", [32, 64, 128])
    clf_dropout = trial.suggest_float("clf_dropout", 0.1, 0.5, step=0.1)

    # -----------------------------
    # Longitud después de phi(.)
    # Conv1d valid: conv_len = Samples - phi_kernel_size + 1
    # AvgPool1d(k=4, s=4): pool_len = floor((conv_len - 4)/4) + 1
    # -----------------------------
    conv_len = samples - phi_kernel_size + 1

    if conv_len < 4:
        raise optuna.TrialPruned(
            f"Trial inválido: conv_len={conv_len} < 4"
        )

    pool_len = (conv_len - 4) // 4 + 1

    if pool_len <= 0:
        raise optuna.TrialPruned(
            f"Trial inválido: pool_len={pool_len} <= 0"
        )

    # -----------------------------
    # Takens ORIGINAL
    # kernel_size = mu + (dx - 1) * tau + 1
    # -----------------------------
    takens_kernel_size = mu + (dx - 1) * tau + 1

    # El kernel Takens debe caber en la longitud disponible
    if takens_kernel_size > pool_len:
        raise optuna.TrialPruned(
            f"Trial inválido: takens_kernel_size={takens_kernel_size} > pool_len={pool_len}"
        )

    # La rama y_sub_t_minus_1 debe caber dentro del kernel original
    if dy * tau > mu + (dx - 1) * tau:
        raise optuna.TrialPruned(
            f"Trial inválido: dy*tau={dy * tau} > mu+(dx-1)*tau={mu + (dx - 1) * tau}"
        )

    return {
        **base_model_args,
        "d_model": d_model,
        "nhead": nhead,
        "num_transformer_layers": num_transformer_layers,
        "dim_feedforward": dim_feedforward,
        "transformer_dropout": transformer_dropout,
        "phi_kernel_size": phi_kernel_size,
        "dx": dx,
        "dy": dy,
        "tau": tau,
        "mu": mu,
        "clf_hidden": clf_hidden,
        "clf_dropout": clf_dropout,
    }


def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "hybridtransformertekte":
        raise ValueError("Este bloque limpio solo soporta model_name='HybridTransformerTEKTE'.")

    set_seed()

    return HybridTransformerTEKTE(
        chans=kwargs["Chans"],
        samples=kwargs["Samples"],
        d_model=kwargs["d_model"],
        nhead=kwargs["nhead"],
        num_transformer_layers=kwargs["num_transformer_layers"],
        dim_feedforward=kwargs["dim_feedforward"],
        transformer_dropout=kwargs["transformer_dropout"],
        phi_kernel_size=kwargs["phi_kernel_size"],
        dx=kwargs["dx"],
        dy=kwargs["dy"],
        tau=kwargs["tau"],
        mu=kwargs["mu"],
        kernel_type=kwargs.get("kernel_type", "rational_quadratic"),
        kernel_amplitude=kwargs.get("kernel_amplitude", 1.0),
        kernel_length_scale=kwargs.get("kernel_length_scale", 1.0),
        kernel_alpha=kwargs.get("kernel_alpha", 1.0),
        trainable_kernel_amplitude=kwargs.get("trainable_kernel_amplitude", False),
        trainable_kernel_length_scale=kwargs.get("trainable_kernel_length_scale", False),
        trainable_kernel_alpha=kwargs.get("trainable_kernel_alpha", False),
        clf_hidden=kwargs["clf_hidden"],
        clf_dropout=kwargs["clf_dropout"],
    )


def suggest_compile_args(trial, model_name, base_compile_args=None):
    if base_compile_args is None:
        base_compile_args = {}

    model_name = model_name.lower()

    if model_name != "hybridtransformertekte":
        raise ValueError("Este bloque limpio solo soporta model_name='HybridTransformerTEKTE'.")

    return {
        **base_compile_args,
        "learning_rate": trial.suggest_categorical("learning_rate", [1e-2, 1e-3, 1e-4]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True),
        "schedule_factor": trial.suggest_float("schedule_factor", 0.1, 0.8),
        "schedule_patience": trial.suggest_int("schedule_patience", 5, 15),
        "min_lr": trial.suggest_float("min_lr", 1e-7, 1e-5, log=True),
    }


def build_compile_config(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "hybridtransformertekte":
        raise ValueError("Este bloque limpio solo soporta model_name='HybridTransformerTEKTE'.")

    compile_args = {
        "optimizer_name": "adam",
        "optimizer_params": {
            "lr": kwargs["learning_rate"],
            "weight_decay": kwargs["weight_decay"],
        },
        "scheduler_name": "ReduceLROnPlateau",
        "scheduler_params": {
            "mode": "min",
            "factor": kwargs["schedule_factor"],
            "patience": kwargs["schedule_patience"],
            "min_lr": kwargs["min_lr"],
        },
        "loss_fn": classification_loss,
    }

    callbacks = []

    return compile_args, callbacks

In [10]:
def ensure_binary_labels(y):
    y = np.asarray(y)

    # ya es vector binario [N]
    if y.ndim == 1:
        return y.astype(np.float32)

    # caso [N, 1]
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1).astype(np.float32)

    # caso one-hot [N, 2] -> convertir a clase 0/1
    if y.ndim == 2 and y.shape[1] == 2:
        return np.argmax(y, axis=1).astype(np.float32)

    raise ValueError(f"Formato de y no soportado: shape={y.shape}")


def _build_optimizer_and_scheduler(model, compile_args):
    optimizer_name = compile_args["optimizer_name"].lower()
    optimizer_params = deepcopy(compile_args["optimizer_params"])

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    scheduler_name = compile_args.get("scheduler_name", None)
    scheduler = None

    if scheduler_name is not None:
        scheduler_name = scheduler_name.lower()
        scheduler_params = deepcopy(compile_args.get("scheduler_params", {}))

        if scheduler_name == "reducelronplateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, **scheduler_params
            )
        else:
            raise ValueError(f"Scheduler no soportado: {scheduler_name}")

    return optimizer, scheduler


def _run_epoch_pytorch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32).view(-1)

        optimizer.zero_grad()

        out = model(xb, return_dict=True, need_attention=False)
        logits = out["logits"]

        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

    return running_loss / max(n_samples, 1)


@torch.no_grad()
def _evaluate_pytorch_binary(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    n_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32).view(-1)

        out = model(xb, return_dict=True, need_attention=False)
        logits = out["logits"]
        probs = torch.sigmoid(logits)

        loss = loss_fn(logits, yb)

        preds = (probs >= 0.5).long()

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(yb.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs).astype(np.float32)
    y_pred = np.concatenate(all_preds).astype(np.int64)
    y_true = np.concatenate(all_true).astype(np.int64)

    mean_loss = running_loss / max(n_samples, 1)

    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_value = np.nan

    metrics = {
        "loss": mean_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    return metrics


def train_L24O_cv(
    model_name,
    X,
    y,
    sbjs,
    folds,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    study_name="study_hybridtransformertekte_tdah",
    trial_number=0,
    trial_cache_root="optuna_resume_cache",
    trial_cache_key=None,
    cache_model_format="weights",   # "weights" o "full"
    force_retrain=False,
    evaluate_test=True,
):
    """
    Entrenamiento Leave-24-Out CV para HybridTransformerTEKTE con reanudación por trial/fold.

    Si evaluate_test=False:
      - no crea test_loader
      - no calcula métricas de test
      - devuelve mean_accuracy=None y mean_metrics=None

    Si evaluate_test=True:
      - calcula validación + test normalmente
    """
    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    saved_model_paths = {}

    model_name = model_name.lower()
    if model_name != "hybridtransformertekte":
        raise ValueError("Esta versión solo soporta 'HybridTransformerTEKTE'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    trial_subdir = trial_cache_key if trial_cache_key is not None else f"trial_{trial_number:04d}"

    trial_dir = os.path.join(
        trial_cache_root,
        study_name,
        trial_subdir
    )
    os.makedirs(trial_dir, exist_ok=True)

    for fold, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        fold_dir = os.path.join(trial_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        checkpoint_path = os.path.join(fold_dir, "resume_checkpoint.pt")
        best_state_path = os.path.join(fold_dir, "best_state.pt")
        fold_info_path = os.path.join(fold_dir, "fold_result.pkl")
        history_path = os.path.join(fold_dir, "history.pkl")

        if cache_model_format == "weights":
            cached_model_path = os.path.join(fold_dir, "final_state_dict.pt")
        elif cache_model_format == "full":
            cached_model_path = os.path.join(fold_dir, "final_model.pt")
        else:
            raise ValueError("cache_model_format debe ser 'weights' o 'full'")

        # --------------------------------------------------
        # Si queremos reentrenar de verdad, borrar caché del fold
        # --------------------------------------------------
        if force_retrain:
            for p in [checkpoint_path, best_state_path, fold_info_path, history_path, cached_model_path]:
                if os.path.exists(p):
                    os.remove(p)

        # --------------------------------------------------
        # Si el fold ya terminó antes, reutilizarlo
        # SOLO si el caché coincide con evaluate_test actual
        # --------------------------------------------------
        if (not force_retrain) and os.path.exists(fold_info_path) and os.path.exists(cached_model_path):
            with open(fold_info_path, "rb") as f:
                fold_info = pickle.load(f)

            cached_eval_mode = fold_info.get("evaluate_test", None)

            if cached_eval_mode == evaluate_test:
                if evaluate_test:
                    all_fold_metrics.append(fold_info["fold_metrics"])
                all_fold_val_metrics.append(fold_info["fold_val_metrics"])
                total_histories.append(fold_info["history"])
                saved_model_paths[fold] = cached_model_path
                print(f"[Trial {trial_number}] Fold {fold}: reutilizado desde disco.")
                continue
            else:
                print(
                    f"[Trial {trial_number}] Fold {fold}: caché incompatible "
                    f"(cached evaluate_test={cached_eval_mode}, actual={evaluate_test}). Reentrenando."
                )

        # --------------------------------------------------
        # Índices train/val/test
        # --------------------------------------------------
        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        val_idx = [i for i, sbj in enumerate(sbjs) if sbj in val_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        y_train = ensure_binary_labels(y_train)
        y_val = ensure_binary_labels(y_val)
        if evaluate_test:
            y_test = ensure_binary_labels(y_test)

        # --------------------------------------------------
        # Semillas
        # --------------------------------------------------
        set_seed(seed + fold)

        # --------------------------------------------------
        # Construcción del modelo
        # --------------------------------------------------
        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold

        model = build_eeg_model(model_name=model_name, **model_args_local).to(device)

        compile_cfg_local = deepcopy(compile_cfg)
        compile_args_local, callbacks = build_compile_config(
            model_name=model_name,
            Chans=model_args["Chans"],
            **compile_cfg_local
        )

        loss_fn = compile_args_local["loss_fn"]
        optimizer, scheduler = _build_optimizer_and_scheduler(model, compile_args_local)

        # --------------------------------------------------
        # DataLoaders
        # --------------------------------------------------
        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )
        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        g = torch.Generator()
        g.manual_seed(seed + fold)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
            generator=g,
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None
        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        # --------------------------------------------------
        # Reanudación del entrenamiento del fold
        # --------------------------------------------------
        start_epoch = 0
        best_val_loss = np.inf
        patience_counter = 0
        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
        }

        if (not force_retrain) and os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

            if scheduler is not None and ckpt.get("scheduler_state_dict", None) is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt["best_val_loss"]
            patience_counter = ckpt["patience_counter"]
            history = ckpt["history"]

            print(f"[Trial {trial_number}] Fold {fold}: reanudando desde epoch {start_epoch}.")

        # --------------------------------------------------
        # Entrenamiento manual PyTorch
        # --------------------------------------------------
        for epoch in range(start_epoch, epochs):
            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_binary(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)

            improved = val_loss < (best_val_loss - 1e-4)
            if improved:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), best_state_path)
            else:
                patience_counter += 1

            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "best_val_loss": best_val_loss,
                "patience_counter": patience_counter,
                "history": history,
            }
            torch.save(checkpoint, checkpoint_path)

            with open(history_path, "wb") as f:
                pickle.dump(history, f)

            if patience_counter >= 25:
                break

        # --------------------------------------------------
        # Restaurar mejores pesos según validación
        # --------------------------------------------------
        if os.path.exists(best_state_path):
            best_state = torch.load(best_state_path, map_location=device)
            model.load_state_dict(best_state)

        # --------------------------------------------------
        # Predicciones en validación
        # --------------------------------------------------
        val_eval = _evaluate_pytorch_binary(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        # --------------------------------------------------
        # Predicciones en test
        # --------------------------------------------------
        if evaluate_test:
            test_eval = _evaluate_pytorch_binary(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }
        else:
            fold_metrics = None

        # --------------------------------------------------
        # Guardar fold terminado
        # --------------------------------------------------
        if cache_model_format == "weights":
            torch.save(model.state_dict(), cached_model_path)
        else:
            torch.save(model, cached_model_path)

        fold_info = {
            "fold_metrics": fold_metrics,
            "fold_val_metrics": fold_val_metrics,
            "history": history,
            "evaluate_test": evaluate_test,
        }

        with open(fold_info_path, "wb") as f:
            pickle.dump(fold_info, f)

        if evaluate_test:
            all_fold_metrics.append(fold_metrics)
        all_fold_val_metrics.append(fold_val_metrics)
        total_histories.append(history)
        saved_model_paths[fold] = cached_model_path

        print(f"[Trial {trial_number}] Fold {fold}: entrenado y guardado.")

    # ------------------------------------------------------
    # Resumen global test
    # ------------------------------------------------------
    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}
        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = np.nanmean(vals)
            mean_metrics[f"std_{key}"] = np.nanstd(vals)

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    # ------------------------------------------------------
    # Resumen global validación
    # ------------------------------------------------------
    mean_val_metrics = {}
    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = np.nanmean(vals)
        mean_val_metrics[f"std_{key}"] = np.nanstd(vals)

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]

    return {
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "saved_model_paths": saved_model_paths,
        "trial_dir": trial_dir,
    }


def train_npz_folds_cv(
    npz_path,
    model_name,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    evaluate_test=True,
):
    """
    Entrena HybridTransformerTEKTE usando folds ya guardados dentro de un archivo .npz.

    Si evaluate_test=False:
      - no calcula test
      - devuelve mean_accuracy=None y mean_metrics=None
    """
    data = np.load(npz_path)

    X = data["X"].astype(np.float32)
    y = ensure_binary_labels(data["y"])

    # --------------------------------------------------
    # Detectar folds disponibles
    # --------------------------------------------------
    fold_numbers = sorted({
        int(key.split("_")[1])
        for key in data.files
        if key.startswith("fold_") and key.endswith("_train_idx")
    })

    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    models = {}

    model_name = model_name.lower()
    if model_name != "hybridtransformertekte":
        raise ValueError("Esta versión solo soporta 'HybridTransformerTEKTE'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold_pos, fold_num in enumerate(fold_numbers):
        train_idx = data[f"fold_{fold_num}_train_idx"]
        test_idx = data[f"fold_{fold_num}_test_idx"]
        val_idx = data[f"fold_{fold_num}_val_idx"]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        y_train = ensure_binary_labels(y_train)
        y_val = ensure_binary_labels(y_val)
        if evaluate_test:
            y_test = ensure_binary_labels(y_test)

        # --------------------------------------------------
        # Semillas
        # --------------------------------------------------
        set_seed(seed + fold_pos)

        # --------------------------------------------------
        # Construcción del modelo
        # --------------------------------------------------
        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold_pos

        model = build_eeg_model(model_name=model_name, **model_args_local).to(device)

        compile_cfg_local = deepcopy(compile_cfg)
        compile_args_local, callbacks = build_compile_config(
            model_name=model_name,
            Chans=model_args["Chans"],
            **compile_cfg_local
        )

        loss_fn = compile_args_local["loss_fn"]
        optimizer, scheduler = _build_optimizer_and_scheduler(model, compile_args_local)

        # --------------------------------------------------
        # DataLoaders
        # --------------------------------------------------
        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )
        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None
        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )
            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        # --------------------------------------------------
        # Entrenamiento manual PyTorch
        # --------------------------------------------------
        best_val_loss = np.inf
        best_state = deepcopy(model.state_dict())
        patience_counter = 0

        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
        }

        for epoch in range(epochs):
            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_binary(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)

            improved = val_loss < (best_val_loss - 1e-4)
            if improved:
                best_val_loss = val_loss
                best_state = deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= 25:
                break

        model.load_state_dict(best_state)
        total_histories.append(history)

        # --------------------------------------------------
        # Predicciones en validación
        # --------------------------------------------------
        val_eval = _evaluate_pytorch_binary(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        all_fold_val_metrics.append(fold_val_metrics)

        # --------------------------------------------------
        # Predicciones en test
        # --------------------------------------------------
        if evaluate_test:
            test_eval = _evaluate_pytorch_binary(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }

            all_fold_metrics.append(fold_metrics)

        models[fold_num] = model

    # ------------------------------------------------------
    # Promedio global test
    # ------------------------------------------------------
    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}
        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = np.nanmean(vals)
            mean_metrics[f"std_{key}"] = np.nanstd(vals)

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    # ------------------------------------------------------
    # Promedio global validación
    # ------------------------------------------------------
    mean_val_metrics = {}
    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = np.nanmean(vals)
        mean_val_metrics[f"std_{key}"] = np.nanstd(vals)

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]

    return {
        "X": X,
        "y": y,
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "models": models,
    }

In [11]:
import os
import shutil
import json
import hashlib

def make_trial_cache_key(model_args, compile_cfg):
    payload = {
        "model_args": model_args,
        "compile_cfg": compile_cfg,
    }
    text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:16]


def make_objective(
    model_name,
    data_mode,
    base_model_args,
    base_compile_args,
    epochs=100,
    batch_size=16,
    seed=42,
    X=None,
    y=None,
    sbjs=None,
    folds=None,
    best_model_dir="best_models",
    save_format="weights",
    direction="maximize",
    resume_root="optuna_resume_cache",
    study_name="study_hybridtransformertekte_tdah",
    force_retrain=False,
):
    def objective(trial):
        model_args = suggest_model_args(
            trial=trial,
            model_name=model_name,
            base_model_args=base_model_args
        )

        compile_cfg = suggest_compile_args(
            trial=trial,
            model_name=model_name,
            base_compile_args=base_compile_args
        )

        trial_cache_key = make_trial_cache_key(model_args, compile_cfg)

        if data_mode != "subject_folds":
            raise ValueError("Esta versión está preparada solo para 'subject_folds' (TDAH).")

        results = train_L24O_cv(
            model_name=model_name,
            X=X,
            y=y,
            sbjs=sbjs,
            folds=folds,
            model_args=model_args,
            compile_cfg=compile_cfg,
            epochs=epochs,
            batch_size=batch_size,
            seed=seed,
            study_name=study_name,
            trial_number=trial.number,
            trial_cache_root=resume_root,
            trial_cache_key=trial_cache_key,
            cache_model_format=save_format,
            force_retrain=force_retrain,
            evaluate_test=False,
        )

        objective_value = float(results["mean_val_accuracy"])

        try:
            prev_best = trial.study.best_value
            has_prev_best = True
        except ValueError:
            prev_best = None
            has_prev_best = False

        if direction == "maximize":
            is_new_best = (not has_prev_best) or (objective_value > prev_best)
        elif direction == "minimize":
            is_new_best = (not has_prev_best) or (objective_value < prev_best)
        else:
            raise ValueError("direction debe ser 'maximize' o 'minimize'")

        if is_new_best:
            if os.path.exists(best_model_dir):
                shutil.rmtree(best_model_dir)

            os.makedirs(best_model_dir, exist_ok=True)

            for fold_id, cached_model_path in results["saved_model_paths"].items():
                if save_format == "weights":
                    dst_path = os.path.join(
                        best_model_dir,
                        f"{model_name}_BESTTRIAL_fold{fold_id}.state_dict.pt"
                    )
                elif save_format == "full":
                    dst_path = os.path.join(
                        best_model_dir,
                        f"{model_name}_BESTTRIAL_fold{fold_id}.model.pt"
                    )
                else:
                    raise ValueError("save_format debe ser 'weights' o 'full'")

                shutil.copy2(cached_model_path, dst_path)

            trial.set_user_attr("saved_as_best", True)
            trial.set_user_attr("best_model_dir", best_model_dir)
        else:
            trial.set_user_attr("saved_as_best", False)

        trial.set_user_attr("model_args", model_args)
        trial.set_user_attr("compile_cfg", compile_cfg)
        trial.set_user_attr("trial_cache_key", trial_cache_key)
        trial.set_user_attr("mean_val_accuracy", objective_value)
        trial.set_user_attr("trial_dir", results["trial_dir"])

        return objective_value

    return objective

In [12]:
def get_or_create_study_local(
    study_name,
    journal_file,
    direction="maximize",
    seed=42,
):
    """
    Crea o reutiliza un estudio Optuna con JournalStorage local.

    Parámetros
    ----------
    study_name : str
        Nombre del estudio.
    journal_file : str
        Ruta al archivo .journal donde se guardará el historial.
    direction : str
        'maximize' o 'minimize'.
    seed : int
        Semilla para el sampler.

    Retorna
    -------
    optuna.study.Study
    """
    journal_dir = os.path.dirname(os.path.abspath(journal_file))
    if journal_dir and not os.path.exists(journal_dir):
        os.makedirs(journal_dir, exist_ok=True)

    storage = JournalStorage(JournalFileBackend(journal_file))

    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction=direction,
        load_if_exists=True,
        sampler=optuna.samplers.GPSampler(
            seed=seed,
            deterministic_objective=True,
        ),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )

    return study

In [13]:
def run_optuna_experiment(
    model_name,
    data_mode,
    study_name,
    journal_file,
    base_model_args,
    base_compile_args,
    epochs=100,
    batch_size=16,
    seed=42,
    X=None,
    y=None,
    sbjs=None,
    folds=None,
    n_trials_total=20,
    best_model_dir="best_models",
    save_format="weights",
    resume_root="optuna_resume_cache",
    force_retrain=False,
):
    objective = make_objective(
        model_name=model_name,
        data_mode=data_mode,
        base_model_args=base_model_args,
        base_compile_args=base_compile_args,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        best_model_dir=best_model_dir,
        save_format=save_format,
        direction="maximize",
        resume_root=resume_root,
        study_name=study_name,
        force_retrain=force_retrain,
    )

    study = get_or_create_study_local(
        study_name=study_name,
        journal_file=journal_file,
        direction="maximize",
        seed=seed,
    )

    # --------------------------------------------------
    # Trials completos vs incompletos
    # --------------------------------------------------
    completed_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    unfinished_trials = [
        t for t in study.trials
        if t.state in (optuna.trial.TrialState.RUNNING, optuna.trial.TrialState.WAITING)
    ]

    # --------------------------------------------------
    # Reencolar trials incompletos con params ya definidos
    # --------------------------------------------------
    requeued_count = 0
    seen_param_signatures = set()

    for t in unfinished_trials:
        if len(t.params) == 0:
            continue

        param_signature = tuple(sorted(t.params.items()))
        if param_signature in seen_param_signatures:
            continue

        study.enqueue_trial(t.params, skip_if_exists=False)
        seen_param_signatures.add(param_signature)
        requeued_count += 1

    # --------------------------------------------------
    # Cuántos trials nuevos faltan realmente
    # --------------------------------------------------
    remaining_completed_needed = max(0, n_trials_total - len(completed_trials))

    print(f"Study name                : {study_name}")
    print(f"Trials completos          : {len(completed_trials)}")
    print(f"Trials incompletos        : {len(unfinished_trials)}")
    print(f"Reencolados               : {requeued_count}")
    print(f"Meta total (completos)    : {n_trials_total}")
    print(f"Faltantes por completar   : {remaining_completed_needed}")

    if remaining_completed_needed > 0:
        study.optimize(
            objective,
            n_trials=remaining_completed_needed,
        )
    else:
        print("El estudio ya alcanzó o superó el número objetivo de trials completos.")

    try:
        print("Best value:", study.best_value)
        print("Best params:", study.best_params)
    except ValueError:
        print("Aún no hay trials completos con best_value disponible.")

    return study

In [14]:
import pickle
import os
import shutil
import json
import hashlib
import optuna
import warnings
import logging

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from optuna.exceptions import ExperimentalWarning
    warnings.filterwarnings("ignore", category=ExperimentalWarning)
except Exception:
    pass

model_name = "HybridTransformerTEKTE"
db = "TDAH"
seed = 42
n_trials_total = 20

study_name = f"study_{model_name}_{db}"

# ==============================
# RUTA DIRECTA DE GUARDADO
# ==============================
SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2"
os.makedirs(SAVE_ROOT, exist_ok=True)

# Carpeta para mejores modelos
BEST_MODEL_DIR = os.path.join(SAVE_ROOT, f"{model_name}_best_models")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Carpeta para caché/reanudación
RESUME_ROOT = os.path.join(SAVE_ROOT, "optuna_resume_cache")
os.makedirs(RESUME_ROOT, exist_ok=True)

# Archivo journal de Optuna
JOURNAL_FILE = os.path.join(SAVE_ROOT, f"{study_name}.journal")

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data()

study = run_optuna_experiment(
    model_name=model_name,
    data_mode="subject_folds",
    study_name=study_name,
    journal_file=JOURNAL_FILE,
    base_model_args={
        "nb_classes": 2,
        "Chans": 19,
                                                                                                                                                                                                                                                                            
    },
    base_compile_args={},
    epochs=100,
    batch_size=16,
    seed=seed,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    n_trials_total=n_trials_total,
    best_model_dir=BEST_MODEL_DIR,
    save_format="weights",
    resume_root=RESUME_ROOT,
    force_retrain=True,
)

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINALIZADO")
print("=" * 80)
print(f"Carpeta principal       : {SAVE_ROOT}")
print(f"Study name              : {study_name}")
print(f"Journal de Optuna       : {JOURNAL_FILE}")
print(f"Mejores modelos         : {BEST_MODEL_DIR}")
print(f"Caché de reanudación    : {RESUME_ROOT}")

if study is not None:
    try:
        print(f"Mejor valor del estudio : {study.best_value}")
        print("Mejores hiperparámetros:")
        for k, v in study.best_trial.params.items():
            print(f"  {k}: {v}")
    except ValueError:
        print("Aún no hay trials completos válidos.")

[I 2026-04-21 17:16:53,561] A new study created in Journal with name: study_HybridTransformerTEKTE_TDAH


Study name                : study_HybridTransformerTEKTE_TDAH
Trials completos          : 0
Trials incompletos        : 0
Reencolados               : 0
Meta total (completos)    : 20
Faltantes por completar   : 20
[Trial 0] Fold 0: entrenado y guardado.
[Trial 0] Fold 1: entrenado y guardado.
[Trial 0] Fold 2: entrenado y guardado.
[Trial 0] Fold 3: entrenado y guardado.


[I 2026-04-21 18:16:59,183] Trial 0 finished with value: 0.6840406004924405 and parameters: {'d_model': 64, 'nhead': 1, 'num_transformer_layers': 1, 'dim_feedforward': 64, 'transformer_dropout': 0.1, 'phi_kernel_size': 123, 'dx': 9, 'dy': 3, 'tau': 1, 'mu': 2, 'clf_hidden': 64, 'clf_dropout': 0.2, 'learning_rate': 0.01, 'weight_decay': 2.9204338471814074e-06, 'schedule_factor': 0.4192489889519252, 'schedule_patience': 13, 'min_lr': 2.50811568604523e-07}. Best is trial 0 with value: 0.6840406004924405.


[Trial 0] Fold 4: entrenado y guardado.
[Trial 1] Fold 0: entrenado y guardado.
[Trial 1] Fold 1: entrenado y guardado.
[Trial 1] Fold 2: entrenado y guardado.
[Trial 1] Fold 3: entrenado y guardado.


[I 2026-04-21 18:41:57,155] Trial 1 finished with value: 0.8493427460861664 and parameters: {'d_model': 64, 'nhead': 1, 'num_transformer_layers': 2, 'dim_feedforward': 64, 'transformer_dropout': 0.1, 'phi_kernel_size': 87, 'dx': 5, 'dy': 2, 'tau': 3, 'mu': 0, 'clf_hidden': 32, 'clf_dropout': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.0007556810141274422, 'schedule_factor': 0.6425929763527802, 'schedule_patience': 15, 'min_lr': 6.161049539380955e-06}. Best is trial 1 with value: 0.8493427460861664.


...

[Trial 17] Fold 4: entrenado y guardado.
[Trial 18] Fold 0: entrenado y guardado.
[Trial 18] Fold 1: entrenado y guardado.
[Trial 18] Fold 2: entrenado y guardado.
[Trial 18] Fold 3: entrenado y guardado.


[I 2026-04-22 00:33:05,602] Trial 18 finished with value: 0.858462900836917 and parameters: {'d_model': 32, 'nhead': 2, 'num_transformer_layers': 2, 'dim_feedforward': 256, 'transformer_dropout': 0.5, 'phi_kernel_size': 21, 'dx': 7, 'dy': 5, 'tau': 2, 'mu': 0, 'clf_hidden': 32, 'clf_dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 5.760463109970659e-07, 'schedule_factor': 0.1, 'schedule_patience': 13, 'min_lr': 2.979764450996565e-07}. Best is trial 16 with value: 0.8868466812300007.


[Trial 18] Fold 4: entrenado y guardado.
[Trial 19] Fold 0: entrenado y guardado.
[Trial 19] Fold 1: entrenado y guardado.
[Trial 19] Fold 2: entrenado y guardado.
[Trial 19] Fold 3: entrenado y guardado.


[I 2026-04-22 00:48:43,460] Trial 19 finished with value: 0.893387014029136 and parameters: {'d_model': 32, 'nhead': 2, 'num_transformer_layers': 2, 'dim_feedforward': 256, 'transformer_dropout': 0.4, 'phi_kernel_size': 99, 'dx': 4, 'dy': 2, 'tau': 5, 'mu': 5, 'clf_hidden': 128, 'clf_dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 1.0394905157104865e-05, 'schedule_factor': 0.1, 'schedule_patience': 11, 'min_lr': 2.4660476577131286e-06}. Best is trial 19 with value: 0.893387014029136.


[Trial 19] Fold 4: entrenado y guardado.
Best value: 0.893387014029136
Best params: {'d_model': 32, 'nhead': 2, 'num_transformer_layers': 2, 'dim_feedforward': 256, 'transformer_dropout': 0.4, 'phi_kernel_size': 99, 'dx': 4, 'dy': 2, 'tau': 5, 'mu': 5, 'clf_hidden': 128, 'clf_dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 1.0394905157104865e-05, 'schedule_factor': 0.1, 'schedule_patience': 11, 'min_lr': 2.4660476577131286e-06}

ENTRENAMIENTO FINALIZADO
Carpeta principal       : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2
Study name              : study_HybridTransformerTEKTE_TDAH
Journal de Optuna       : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2/study_HybridTransformerTEKTE_TDAH.journal
Mejores modelos         : /home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2/HybridTransformerTEKTE_best_models
Caché de reanudación    : /home/usuario-utp/Desktop/Yessica Al

In [16]:
import os
import json
import pickle
import numpy as np

# ==============================
# 1) Cargar estudio
# ==============================
study = get_or_create_study_local(
    study_name=f"study_{model_name}_{db}",
    journal_file=JOURNAL_FILE,
    direction="maximize",
    seed=seed,
)

print("Best value (val):", study.best_value)
print("Best trial number:", study.best_trial.number)
print("Best params:", study.best_trial.params)
print("Best trial user_attrs:", study.best_trial.user_attrs)

# ==============================
# 2) Reconstruir args del mejor trial
# ==============================
best_trial = study.best_trial

model_args = suggest_model_args(
    trial=best_trial,
    model_name=model_name,
    base_model_args={
        "nb_classes": 2,
        "Chans": 19,
        "Samples": 512,
        "alpha": 2,
    }
)

compile_cfg = suggest_compile_args(
    trial=best_trial,
    model_name=model_name,
    base_compile_args={}
)

# importante: verificar que exista trial_cache_key
if "trial_cache_key" not in best_trial.user_attrs:
    raise KeyError(
        "best_trial.user_attrs no contiene 'trial_cache_key'. "
        "Así train_L24O_cv podría no encontrar la caché correcta."
    )

trial_cache_key = best_trial.user_attrs["trial_cache_key"]

# ==============================
# 3) Recuperar resultados del mejor trial
#    (debería reutilizar caché)
# ==============================
results = train_L24O_cv(
    model_name=model_name,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    model_args=model_args,
    compile_cfg=compile_cfg,
    epochs=100,
    batch_size=16,
    seed=seed,
    study_name=f"study_{model_name}_{db}",
    trial_number=best_trial.number,
    trial_cache_root=RESUME_ROOT,
    trial_cache_key=trial_cache_key,
    cache_model_format="weights",
    force_retrain=False,   # si tu función lo soporta
)

# ==============================
# 4) Imprimir accuracy test y std
# ==============================
print("\n" + "=" * 80)
print("MEJOR TRIAL - RESULTADOS EN TEST")
print("=" * 80)

print(f"Accuracy medio en test : {results['mean_accuracy']:.4f}")

if "mean_metrics" in results and "std_accuracy" in results["mean_metrics"]:
    print(f"STD accuracy en test   : {results['mean_metrics']['std_accuracy']:.4f}")

print("\nAccuracy por fold:")
for i, fm in enumerate(results["fold_metrics"]):
    print(f"  Fold {i}: {fm['accuracy']:.4f}")

print("\nResumen test:")
for k, v in results["mean_metrics"].items():
    print(f"  {k}: {v:.4f}")

[I 2026-04-22 08:52:45,750] Using an existing study with name 'study_HybridTransformerTEKTE_TDAH' instead of creating a new one.


Best value (val): 0.893387014029136
Best trial number: 19
Best params: {'d_model': 32, 'nhead': 2, 'num_transformer_layers': 2, 'dim_feedforward': 256, 'transformer_dropout': 0.4, 'phi_kernel_size': 99, 'dx': 4, 'dy': 2, 'tau': 5, 'mu': 5, 'clf_hidden': 128, 'clf_dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 1.0394905157104865e-05, 'schedule_factor': 0.1, 'schedule_patience': 11, 'min_lr': 2.4660476577131286e-06}
Best trial user_attrs: {'saved_as_best': True, 'best_model_dir': '/home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2/HybridTransformerTEKTE_best_models', 'model_args': {'nb_classes': 2, 'Chans': 19, 'Samples': 512, 'd_model': 32, 'nhead': 2, 'num_transformer_layers': 2, 'dim_feedforward': 256, 'transformer_dropout': 0.4, 'phi_kernel_size': 99, 'dx': 4, 'dy': 2, 'tau': 5, 'mu': 5, 'clf_hidden': 128, 'clf_dropout': 0.1}, 'compile_cfg': {'learning_rate': 0.001, 'weight_decay': 1.0394905157104865e-05, 'schedule_factor': 0.1, 'schedule_pa

In [17]:
import os
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================================================
# 1) CONFIGURACIÓN GENERAL
# =========================================================
model_name = "HybridTransformerTEKTE"
db = "TDAH"

# Nueva ruta de resultados basada en el nuevo estudio
SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_hybridtransformer_tekte_tdah_2_repeated"
os.makedirs(SAVE_ROOT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_ROOT, "repeated_test_results.csv")
RESULTS_JSON = os.path.join(SAVE_ROOT, "repeated_test_summary.json")

# 10 repeticiones -> cada seed corre los 5 folds
seeds = list(range(10))

# Puedes dejar esto configurable
epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# 2) MEJORES HIPERPARÁMETROS DEL NUEVO ESTUDIO
#    Best trial = 19
# =========================================================
best_params = {
    "d_model": 32,
    "nhead": 2,
    "num_transformer_layers": 2,
    "dim_feedforward": 256,
    "transformer_dropout": 0.4,
    "phi_kernel_size": 99,
    "dx": 4,
    "dy": 2,
    "tau": 5,
    "mu": 5,
    "clf_hidden": 128,
    "clf_dropout": 0.1,
    "learning_rate": 0.001,
    "weight_decay": 1.0394905157104865e-05,
    "schedule_factor": 0.1,
    "schedule_patience": 11,
    "min_lr": 2.4660476577131286e-06,
}

# =========================================================
# 3) FUNCIONES AUXILIARES
# =========================================================
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def summarize_metric(values):
    arr = np.array(values, dtype=float)
    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "n": int(len(arr)),
    }

# =========================================================
# 4) CARGAR DATOS Y FOLDS
# =========================================================
with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data()

# =========================================================
# 5) ARMAR model_args Y compile_cfg
#    usando best_params fijos
# =========================================================
model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
    "alpha": 2,

    "d_model": best_params["d_model"],
    "nhead": best_params["nhead"],
    "num_transformer_layers": best_params["num_transformer_layers"],
    "dim_feedforward": best_params["dim_feedforward"],
    "transformer_dropout": best_params["transformer_dropout"],
    "phi_kernel_size": best_params["phi_kernel_size"],
    "dx": best_params["dx"],
    "dy": best_params["dy"],
    "tau": best_params["tau"],
    "mu": best_params["mu"],
    "clf_hidden": best_params["clf_hidden"],
    "clf_dropout": best_params["clf_dropout"],
}

compile_cfg = {
    "learning_rate": best_params["learning_rate"],
    "weight_decay": best_params["weight_decay"],
    "schedule_factor": best_params["schedule_factor"],
    "schedule_patience": best_params["schedule_patience"],
    "min_lr": best_params["min_lr"],
}

# =========================================================
# 6) REPETICIONES
#    train_L24O_cv corre todos los folds.
#    Cada seed genera métricas para los 5 folds.
# =========================================================
all_rows = []
all_run_summaries = []

for rep_id, seed in enumerate(seeds):
    print("\n" + "=" * 90)
    print(f"REPETICIÓN {rep_id+1}/{len(seeds)} | seed={seed}")
    print("=" * 90)

    set_all_seeds(seed)

    rep_cache_root = os.path.join(SAVE_ROOT, f"repeat_seed_{seed}")
    os.makedirs(rep_cache_root, exist_ok=True)

    results = train_L24O_cv(
        model_name=model_name,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        model_args=model_args,
        compile_cfg=compile_cfg,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        study_name=f"fixed_best_params_{model_name}_{db}_study2",
        trial_number=rep_id,
        trial_cache_root=rep_cache_root,
        trial_cache_key=f"fixedparams_study2_seed_{seed}",
        cache_model_format=save_format,
        force_retrain=force_retrain,
        evaluate_test=True,   # IMPORTANTE: este script resume métricas de TEST
    )

    fold_metrics = results["fold_metrics"]

    for fold_id, fm in enumerate(fold_metrics):
        row = {
            "seed": seed,
            "repeat_id": rep_id,
            "fold": fold_id,
            "accuracy": float(fm["accuracy"]),
            "recall": float(fm["recall"]),
            "precision": float(fm["precision"]),
            "kappa": float(fm["kappa"]),
            "auc": float(fm["auc"]),
        }
        all_rows.append(row)

    run_summary = {
        "seed": seed,
        "repeat_id": rep_id,
        "mean_accuracy_across_5_folds": float(results["mean_accuracy"]),
        "mean_metrics": {
            k: float(v) for k, v in results["mean_metrics"].items()
        },
    }
    all_run_summaries.append(run_summary)

# =========================================================
# 7) GUARDAR RESULTADOS CRUDOS
# =========================================================
df = pd.DataFrame(all_rows)
df.to_csv(RESULTS_CSV, index=False)

print("\nArchivo CSV guardado en:")
print(RESULTS_CSV)

# =========================================================
# 8) RESUMEN GLOBAL SOBRE LAS 50 CORRIDAS
#    (5 folds x 10 seeds)
# =========================================================
summary_global = {
    "accuracy": summarize_metric(df["accuracy"].tolist()),
    "recall": summarize_metric(df["recall"].tolist()),
    "precision": summarize_metric(df["precision"].tolist()),
    "kappa": summarize_metric(df["kappa"].tolist()),
    "auc": summarize_metric(df["auc"].tolist()),
}

# =========================================================
# 9) RESUMEN POR FOLD
# =========================================================
summary_by_fold = {}
for fold_id in sorted(df["fold"].unique()):
    dff = df[df["fold"] == fold_id]
    summary_by_fold[f"fold_{fold_id}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }

# =========================================================
# 10) RESUMEN POR SEED
# =========================================================
summary_by_seed = {}
for seed in sorted(df["seed"].unique()):
    dff = df[df["seed"] == seed]
    summary_by_seed[f"seed_{seed}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }

# =========================================================
# 11) GUARDAR JSON FINAL
# =========================================================
final_summary = {
    "model_name": model_name,
    "database": db,
    "best_params": best_params,
    "n_folds": 5,
    "n_repetitions": len(seeds),
    "total_test_runs": int(len(df)),
    "global_summary_over_50_runs": summary_global,
    "summary_by_fold": summary_by_fold,
    "summary_by_seed": summary_by_seed,
    "run_summaries": all_run_summaries,
    "csv_path": RESULTS_CSV,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("\nArchivo JSON guardado en:")
print(RESULTS_JSON)

# =========================================================
# 12) IMPRIMIR RESULTADOS FINALES
# =========================================================
print("\n" + "=" * 90)
print("RESULTADOS FINALES - 5 FOLDS x 10 REPETICIONES = 50 TESTS")
print("=" * 90)

print(f"Accuracy  : {summary_global['accuracy']['mean']:.4f} ± {summary_global['accuracy']['std_sample']:.4f}")
print(f"Recall    : {summary_global['recall']['mean']:.4f} ± {summary_global['recall']['std_sample']:.4f}")
print(f"Precision : {summary_global['precision']['mean']:.4f} ± {summary_global['precision']['std_sample']:.4f}")
print(f"Kappa     : {summary_global['kappa']['mean']:.4f} ± {summary_global['kappa']['std_sample']:.4f}")
print(f"AUC       : {summary_global['auc']['mean']:.4f} ± {summary_global['auc']['std_sample']:.4f}")

print("\nResumen por fold:")
for fold_name, vals in summary_by_fold.items():
    print(f"{fold_name}: Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f}")

print("\nResumen por seed:")
for seed_name, vals in summary_by_seed.items():
    print(f"{seed_name}: Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f}")


REPETICIÓN 1/10 | seed=0
[Trial 0] Fold 0: entrenado y guardado.
[Trial 0] Fold 1: entrenado y guardado.
[Trial 0] Fold 2: entrenado y guardado.
[Trial 0] Fold 3: entrenado y guardado.
[Trial 0] Fold 4: entrenado y guardado.

REPETICIÓN 2/10 | seed=1
[Trial 1] Fold 0: entrenado y guardado.
[Trial 1] Fold 1: entrenado y guardado.
[Trial 1] Fold 2: entrenado y guardado.
[Trial 1] Fold 3: entrenado y guardado.
[Trial 1] Fold 4: entrenado y guardado.

REPETICIÓN 3/10 | seed=2
[Trial 2] Fold 0: entrenado y guardado.
[Trial 2] Fold 1: entrenado y guardado.
[Trial 2] Fold 2: entrenado y guardado.
[Trial 2] Fold 3: entrenado y guardado.
[Trial 2] Fold 4: entrenado y guardado.

REPETICIÓN 4/10 | seed=3
[Trial 3] Fold 0: entrenado y guardado.
[Trial 3] Fold 1: entrenado y guardado.
[Trial 3] Fold 2: entrenado y guardado.
[Trial 3] Fold 3: entrenado y guardado.
[Trial 3] Fold 4: entrenado y guardado.

REPETICIÓN 5/10 | seed=4
[Trial 4] Fold 0: entrenado y guardado.
[Trial 4] Fold 1: entrenado y 